### Checkpoint 1

Implement your algorithm to process and annotate the traces of the 20 unipolar electrogram signals, identifying focal activity based on activation patterns and signal morphology.

The algorithm must return the following outputs:

1. **Activation Annotations**: for each of the 20 signals, generate a list of activation times. These activation times correspond to the local minima of the first derivative of the signal, indicating the moments of depolarization at each electrode.
2. **Focal proximity**: assign a score of 1 to the sensor closest to the focal activity and 0 to all others. If the focal activity is within two sensors (up to a specified tolerance), assign 0.5 to each of the two sensors.
This output should be guided by signal morphology: focal activity is expected to correspond to a QS morphology (i.e., a single dominant negative deflection).

Provide your algorithm's implementation in this notebook to annotate the traces of the 20 unipolar signals and identify focal activity based on activation patterns and waveform morphology.



In [ ]:
#Imports
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# load data
signals = np.load('checkpoint1dataset.npy')


In [ ]:
# select event to be display
ind_event = 0

# select a spline (options: 0,1,2,3,4)
ind_spline = 3

# plot signals

plt.figure(1+ind_spline)
countEL = 1
for indS in np.arange(ind_spline*4,(ind_spline+1)*4,1):
    plt.subplot(4,1,countEL)
            
    plt.plot(signals[ind_event,indS,:])
    plt.grid(True)

    countEL = countEL + 1

plt.show()

signals_event = signals[ind_event,:,:]

In [ ]:
def GroupSolutionCP1(signals_event):
    n_sensors, n_samples = signals_event.shape
    timings = []
    
    min_dist = 100 
    time_tol = 4
    
    act_times = [[], []] 
    qs_flags = [[], []]

    # 1. EFFICIENZA: Derivata calcolata su tutta la matrice in modo vettorizzato
    ds_all = np.diff(signals_event, axis=1)
    
    for i in range(n_sensors):
        sig = signals_event[i, :]
        ds = ds_all[i, :]
        
        # Troviamo i minimi locali
        min_indices = argrelextrema(ds, np.less)[0]
        
        selected = []
        qs_status = [False, False]
        
        if len(min_indices) > 0:
            depths = ds[min_indices]
            
            # Ordiniamo dal minimo più profondo (pendenza più negativa)
            sorted_args = np.argsort(depths)
            candidates = min_indices[sorted_args]
            
            # Selezioniamo fino a 2 attivazioni distanti almeno min_dist
            for idx in candidates:
                if len(selected) == 0:
                    selected.append(idx)
                elif len(selected) == 1:
                    if abs(idx - selected[0]) >= min_dist:
                        selected.append(idx)
                        break
            
            # Ordinamento finale per garantire che siano in ordine temporale
            selected.sort()
            
            # 2. MORFOLOGIA QS AFFINATA
            for j, val in enumerate(selected):
                # Finestra ristretta per evitare overlap con onde T/P vicine
                window_start = max(0, val - 20)
                window_end = min(n_samples, val + 25)
                
                # Calcolo baseline sui primissimi 5 campioni della finestra
                baseline = np.mean(sig[window_start : window_start + 5])
                
                # Calcolo del picco positivo pre-attivazione e di quello negativo
                max_pos = np.max(sig[window_start : val]) - baseline
                max_neg = np.min(sig[val : window_end]) - baseline
                
                # Un vero QS non ha un'onda R (positiva) pre-attivazione significativa
                # Tolleranza al 15% per gestire leggeri rumori o derive della linea di base
                if max_pos < 0.15 * abs(max_neg):
                    qs_status[j] = True

        # Formattazione array di output
        row_time = [None, None]
        for j in range(len(selected)):
            row_time[j] = int(selected[j])
            
        row_final = [t if t is not None else np.nan for t in row_time]
        timings.append(row_final)

        # Salvataggio in colonna per calcolo focale
        for j in range(2):
            t_val = row_final[j] if not np.isnan(row_final[j]) else np.inf
            act_times[j].append(t_val)
            qs_flags[j].append(qs_status[j])

    # 3. PROSSIMITÀ FOCALE (LS) CON FALLBACK SALVA-CRASH
    LS_array = np.zeros((n_sensors, 2))

    for j in range(2):
        times = np.array(act_times[j])
        flags = np.array(qs_flags[j])
        
        # Se c'è almeno un tempo valido per questo battito
        if np.any(np.isfinite(times)):
            
            # Troviamo chi ha il flag QS
            qs_candidates = [idx for idx, f in enumerate(flags) if f and np.isfinite(times[idx])]
            
            # FALLBACK: Se nessun sensore ha QS (segnale anomalo), prendiamo tutti quelli validi 
            # per non far crashare l'algoritmo e assegnare comunque un focale all'attivazione più precoce.
            if not qs_candidates:
                qs_candidates = [idx for idx, t in enumerate(times) if np.isfinite(t)]
                
            if qs_candidates:
                # Ordiniamo i candidati dal più precoce al più tardivo
                qs_candidates.sort(key=lambda x: times[x])
                
                best_idx = qs_candidates[0]
                t0 = times[best_idx]
                
                if len(qs_candidates) == 1:
                    LS_array[best_idx, j] = 1.0
                else:
                    second_idx = qs_candidates[1]
                    t1 = times[second_idx]
                    
                    # Logica di Tolleranza: se il secondo è entro 4 campioni, diamo 0.5 a entrambi
                    if abs(t1 - t0) <= time_tol:
                        LS_array[best_idx, j] = 0.5
                        LS_array[second_idx, j] = 0.5
                    else:
                        LS_array[best_idx, j] = 1.0

    return timings, LS_array.tolist()

    

In [ ]:
# test it

timings_event,LS_event =  GroupSolutionCP1( signals_event )

print(timings_event)